> **Public evidence copy:** Stored outputs and execution counts have been removed. 
> The participant-level workbook is not included. Set `JAI_SURVEY_DATA` to an 
> authorized secure copy before running this notebook.


# Dog vaccination and sterilisation

## Setup

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px 
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import os 

In [ ]:
from pathlib import Path

data_path_value = os.environ.get("JAI_SURVEY_DATA")
if not data_path_value:
    raise RuntimeError(
        "Set JAI_SURVEY_DATA to the absolute path of the restricted survey workbook. "
        "Do not copy participant data into this repository."
    )

file = Path(data_path_value).expanduser().resolve()
if not file.is_file():
    raise FileNotFoundError(f"Restricted survey workbook not found: {file}")



In [ ]:
df1 = pd.read_excel(file, 'Jai KAP Survey 2025')
df3 = pd.read_excel(file, 'dog_details_current')


In [ ]:
df1_filtered = df1[
    (df1['Is someone >18 present at the household?'] == 'Yes') &
    (df1['Has the respondent consented to undertake the survey?'] == 'Yes')
]



In [ ]:
df1_ong = df1_filtered[df1_filtered["District"] == "Ongkharak"]
df1_pak = df1_filtered[df1_filtered["District"] == "Pak Phli"]


> Records without consent or without an eligible adult respondent are excluded according to the analysis cleaning rule.


## Analysis


In [ ]:
# merge df3._parent_index with df1._index 

In [ ]:
# --- 1) Merge District into df3 via parent/child keys ---
# Ensure keys are same dtype
df1_filtered["_index"] = df1_filtered["_index"].astype(str)
df3["_parent_index"]   = df3["_parent_index"].astype(str)

# Keep only the columns we need from df1
keymap = df1_filtered[["_index", "District"]].drop_duplicates()

# Merge: each dog (df3) gets its household's District
df3m = df3.merge(keymap, left_on="_parent_index", right_on="_index", how="left")

# Optional: drop rows without a matched district
df3m = df3m.dropna(subset=["District"])

# --- 2) Q33–34: Where did you acquire this dog? (stacked %) ---
col = "Where did you acquire this dog?"

# If you want to tidy up NAs:
df3m = df3m[df3m[col].notna()]

acq_counts = df3m.groupby("District")[col].value_counts().unstack(fill_value=0)
acq_percent = acq_counts.div(acq_counts.sum(axis=1), axis=0) * 100

# Optional: enforce district order
acq_percent = acq_percent.reindex(index=["Pak Phli", "Ongkharak"])

# Plot
fig = go.Figure()
for source in acq_percent.columns:
    fig.add_trace(go.Bar(
        x=acq_percent.index,     # Districts
        y=acq_percent[source],   # %
        name=source
    ))

fig.update_layout(
    barmode="stack",
    title="Where Did You Acquire This Dog?",
    xaxis_title="District",
    yaxis_title="Percent of Owned Dogs",
    height=600,
    width=1200,
    legend_title_text="Source"
)
fig.show()


In [ ]:
import plotly.graph_objects as go

# Vaccination status counts
col = "Was this dog vaccinated against rabies in the last 12 months?"
vax_counts = df3[col].value_counts(dropna=False)

# Pie chart
fig = go.Figure(
    data=[go.Pie(
        labels=vax_counts.index,
        values=vax_counts.values,
        hole=0.4,
        textinfo="percent+label"
    )]
)
fig.update_layout(
    title="Rabies Vaccination Status (All Dogs)",
    width=800,
    height=600
)
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

col = "Was this dog vaccinated against rabies in the last 12 months?"

def norm_vax(x):
    if pd.isna(x): 
        return "No response"
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}:  return "Yes"
    if s in {"no", "n", "false", "0"}:  return "No"
    return "Other"  # catch any odd values

df3["vax_norm"] = df3[col].apply(norm_vax)

# sanity check the unique raw values (helps spot hidden spaces)
print(df3[col].value_counts(dropna=False))
print("\nNormalized:")
print(df3["vax_norm"].value_counts())



In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

col_vax = "Was this dog vaccinated against rabies in the last 12 months?"

# --- Merge District info ---
df1_filtered["_index"] = df1_filtered["_index"].astype(str)
df3["_parent_index"]   = df3["_parent_index"].astype(str)

keymap = df1_filtered[["_index", "District"]].drop_duplicates()
df3m = df3.merge(keymap, left_on="_parent_index", right_on="_index", how="left").dropna(subset=["District"])

# --- Plot setup ---
district_order = ["Pak Phli", "Ongkharak"]
status_order   = ["Yes", "No"]
color_map      = {"Yes": "#54C788", "No": "#d75b4f"}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=("Pak Phli (CNVR)", "Ongkharak (Non-CNVR)")
)

# --- Add the pies ---
for i, d in enumerate(district_order, start=1):
    g = df3m[df3m["District"] == d]
    counts = g[col_vax].fillna("No response").value_counts()
    for k in status_order:
        if k not in counts:
            counts[k] = 0
    counts = counts[status_order]

    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.6,
            textinfo="percent+label",
            marker=dict(colors=[color_map[k] for k in counts.index]),
            hovertemplate="%{label}: %{value} dogs (%{percent})<extra></extra>",
            name=d
        ),
        row=1, col=i
    )

# --- Add labels just above each donut (inside plot area) ---
annotations = []
for i, d in enumerate(district_order):
    annotations.append(dict(
        text=f"<b>{d}</b>",
        x=0.225 if i == 0 else 0.775,  # align with each pie center
        y=0.5,                      # position slightly above donut
        showarrow=False,
        font=dict(size=16)
    ))

fig.update_layout(
    title_text="Rabies Vaccination Status by District",
    title_x=0.5,
    height=650,
    width=1200,
    legend_title_text="Status",
    annotations=annotations
)

fig.show()

for i, d in enumerate(district_order, start=1):
    g = df3m[df3m["District"] == d]
    counts = g[col_vax].fillna("No response").value_counts()
    for k in status_order:
        if k not in counts:
            counts[k] = 0
    counts = counts[status_order]
    print(counts)

In [ ]:
col_ster = "Is this dog sterilised?"

# Count and percent by district
ster_counts = df3m.groupby("District")[col_ster].value_counts().unstack(fill_value=0)
ster_percent = ster_counts.div(ster_counts.sum(axis=1), axis=0) * 100
display(ster_counts, ster_percent)


In [ ]:
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Merge district info ---
df1["_index"] = df1["_index"].astype(str)
df3["_parent_index"] = df3["_parent_index"].astype(str)

keymap = df1[["_index", "District"]].drop_duplicates()
df3m = df3.merge(keymap, left_on="_parent_index", right_on="_index", how="left")

# Drop dogs without district info
df3m = df3m.dropna(subset=["District"])


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

col_ster = "Is this dog sterilised?"

# --- 1) Normalize sterilisation responses to Yes / No / Don't know ---
def norm_resp(x: str) -> str:
    if pd.isna(x):
        return "Don't know"
    s = str(x).strip()
    s = s.replace("’", "'")  # normalize curly apostrophes
    s_low = s.lower()
    if s_low == "yes":
        return "Yes"
    if s_low == "no":
        return "No"
    if s_low in {"don't know", "dont know", "don’t know"}:
        return "Don't know"
    # Any other text (e.g., accidental free text) -> treat as Don't know
    return "Don't know"

df3m["_ster_norm"] = df3m[col_ster].apply(norm_resp)

# Optional sanity check
# print(pd.crosstab(df3m["District"], df3m["_ster_norm"]))

# --- 2) Donut charts by district ---
district_order = ["Pak Phli", "Ongkharak"]
label_order    = ["Yes", "No", "Don't know"]
color_map      = {"Yes": "#73c699", "No": "#d16443", "Don't know": "#BDBDBD"}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=("Pak Phli (CNVR)", "Ongkharak (Non-CNVR)")
)

for i, d in enumerate(district_order, start=1):
    g = df3m[df3m["District"] == d]
    counts = g["_ster_norm"].value_counts()
    counts = counts.reindex(label_order, fill_value=0)

    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.6,
            marker=dict(colors=[color_map[k] for k in counts.index]),
            textinfo="percent+label",
            name=d,
            hovertemplate="%{label}: %{value} dogs (%{percent})<extra></extra>"
        ),
        row=1, col=i
    )

# --- 3) Add district labels + sample sizes inside each donut ---
for tr in fig.data:
    if tr.type == "pie":
        x_center = sum(tr.domain["x"]) / 2
        y_center = sum(tr.domain["y"]) / 2
        n = int(sum(tr.values))
        fig.add_annotation(
            x=x_center, y=y_center,
            text=f"<b>{tr.name}</b><br>",
            showarrow=False, font=dict(size=13, color="#333")
        )

fig.update_layout(
    title="Dog Sterilisation Status by District",
    height=600, width=1100,
    legend_title_text="Status"
)

fig.show()


In [ ]:
col_ster = "Is this dog sterilised?"

# --- 1) Normalize sterilisation responses to Yes / No / Don't know ---
def norm_resp(x: str) -> str:
    if pd.isna(x):
        return "Don't know"
    s = str(x).strip().replace("’", "'")  # normalize curly apostrophes
    s_low = s.lower()
    if s_low == "yes":
        return "Yes"
    if s_low == "no":
        return "No"
    if s_low in {"don't know", "dont know", "don’t know"}:
        return "Don't know"
    return "Don't know"

df3m["_ster_norm"] = df3m[col_ster].apply(norm_resp)

# --- 2) Donut charts by district (match previous style) ---
district_order = ["Pak Phli", "Ongkharak"]
label_order    = ["Yes", "No", "Don't know"]

# Match previous green/red and add gray for unknown
color_map = {"Yes": "#54C788", "No": "#d75b4f", "Don't know": "#BDBDBD"}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
    subplot_titles=("Pak Phli (CNVR)", "Ongkharak (Non-CNVR)")
)

for i, d in enumerate(district_order, start=1):
    g = df3m[df3m["District"] == d]
    counts = g["_ster_norm"].value_counts().reindex(label_order, fill_value=0)

    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.6,  # <-- match previous donut width
            marker=dict(colors=[color_map[k] for k in counts.index]),
            textinfo="percent+label",
            name=d,
            hovertemplate="%{label}: %{value} dogs (%{percent})<extra></extra>"
        ),
        row=1, col=i
    )

# --- 3) District labels ABOVE each donut (match previous layout) ---
annotations = []
for i, d in enumerate(district_order):
    annotations.append(dict(
        text=f"<b>{d}</b>",
        x=0.225 if i == 0 else 0.775,   # align with each pie center like before
        y=0.5,                        # slightly above donut
        showarrow=False,
        font=dict(size=16)
    ))

fig.update_layout(
    title_text="Dog Sterilisation Status by District",
    title_x=0.5,
    height=650,
    width=1200,
    legend_title_text="Status",
    annotations=annotations
)

# Ensure both pies share the same hole size (belt & suspenders)
fig.update_traces(hole=0.6, selector=dict(type='pie'))

fig.show()


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

col_ster = "Is this dog sterilised?"

# --- Normalize sterilisation responses ---
def norm_resp(x: str) -> str:
    if pd.isna(x):
        return "Don't know"
    s = str(x).strip().replace("’", "'").lower()
    if s == "yes":
        return "Yes"
    elif s == "no":
        return "No"
    elif s in {"don't know", "dont know", "don’t know"}:
        return "Don't know"
    return "Don't know"

df3m["_ster_norm"] = df3m[col_ster].apply(norm_resp)

# --- Plot donuts ---
district_order = ["Pak Phli", "Ongkharak"]
label_order    = ["Yes", "No", "Don't know"]
color_map      = {"Yes": "#73c699", "No": "#d16443", "Don't know": "#BDBDBD"}

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "pie"}]],
)

for i, d in enumerate(district_order, start=1):
    g = df3m[df3m["District"] == d]
    counts = g["_ster_norm"].value_counts().reindex(label_order, fill_value=0)

    fig.add_trace(
        go.Pie(
            labels=counts.index,
            values=counts.values,
            hole=0.4,
            marker=dict(colors=[color_map[k] for k in counts.index]),
            textinfo="percent+label",
            name=d,
            hovertemplate="%{label}: %{value} dogs (%{percent})<extra></extra>"
        ),
        row=1, col=i
    )

# --- Add annotations above donuts ---
annotations = []
for i, d in enumerate(district_order):
    annotations.append(dict(
        text=f"<b>{d}</b>",
        x=0.175 if i == 0 else 0.85,  # align with each pie center
        y=1.08,                    # slightly above the donut
        showarrow=False,
        font=dict(size=16)
    ))

fig.update_layout(
    title="Dog Sterilisation Status by District",
    height=600, width=1100,
    legend_title_text="Status",
    annotations=annotations
)

fig.show()


In [ ]:
# --- Print summary for both districts ---
print("=== Sterilisation Status by District ===\n")

summary = []
for d in district_order:
    g = df3m[df3m["District"] == d]
    counts = g["_ster_norm"].value_counts().reindex(label_order, fill_value=0)
    total = counts.sum()

    print(f"{d}:")
    print(counts)
    print(f"Total dogs recorded: {total}\n")

    # store summary for optional DataFrame
    for status, value in counts.items():
        summary.append({"District": d, "Status": status, "Count": value, "Percent": round((value / total) * 100, 1)})

# Optional: show as table
df_summary = pd.DataFrame(summary)
print("Summary table:\n", df_summary)


In [ ]:
df3m["_ster_norm"].value_counts()

> Free-text reason fields are intentionally excluded from this public notebook because verbatim responses and rare categories may create disclosure risk.
